# SCVAE-XC emotion demo: Colab + FastAPI + Gradio

**Architecture:** Gradio frontend → HTTP → FastAPI backend → trained SCVAE-XC model.

There is one model per test culture (Chinese, German, French), each trained on the other two cultures; pick it with the **Test culture** dropdown.

Run the cells in order (Runtime → Run all). A GPU is not needed: prediction takes milliseconds on CPU.

## 1. Install packages

In [ ]:
!pip -q install fastapi uvicorn pyngrok gradio requests matplotlib

## 2. Upload the demo folder
Upload **`SCVAE-XC_Demo.zip`** (from your Downloads folder) when the button appears. Any older copy in Colab is deleted first, so you always get the version you upload.

**If you change the zip later:** Runtime → *Disconnect and delete runtime*, then Run all again.

In [ ]:
from google.colab import files
import os, shutil, glob
shutil.rmtree('/content/SCVAE-XC_Demo', ignore_errors=True)
for z in glob.glob('/content/SCVAE-XC_Demo*.zip'):
    os.remove(z)
up = files.upload()  # choose SCVAE-XC_Demo.zip
zip_name = list(up)[0]
!unzip -o -q "$zip_name" -d /content
%cd /content/SCVAE-XC_Demo
assert os.path.isdir('models'), 'This zip is an old version (no models/ folder). Upload the new SCVAE-XC_Demo.zip.'
!ls models

## 3. Start the FastAPI backend (port 8000, in the background)

In [ ]:
import sys, threading, time, socket, requests, uvicorn
for m in ('api', 'app', 'scvae_xc_model'):  # forget code loaded by an earlier run
    sys.modules.pop(m, None)

# a fresh port each run, so a backend left over from an earlier run is never reused
with socket.socket() as s_:
    s_.bind(('', 0)); API_PORT = s_.getsockname()[1]
server = uvicorn.Server(uvicorn.Config('api:app', host='0.0.0.0', port=API_PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
LOCAL_API = f'http://127.0.0.1:{API_PORT}'
for _ in range(120):
    try:
        print(requests.get(LOCAL_API + '/cultures', timeout=1).json()); break
    except requests.RequestException:
        time.sleep(0.5)

## 4. (Optional) Public API URL with ngrok
Only needed if something *outside* Colab should call the API (for example a Flask app on your laptop, or showing `/docs` to the class). Get a free token at https://dashboard.ngrok.com and paste it below. Leave it empty to skip.

In [ ]:
NGROK_TOKEN = ''  # paste your ngrok authtoken here, or leave empty
API_URL = LOCAL_API
if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    API_URL = ngrok.connect(API_PORT).public_url
    print('Public FastAPI URL:', API_URL, '\nInteractive docs:', API_URL + '/docs')
else:
    print('Using the local API inside Colab:', API_URL)

## 5. Try the API directly

In [ ]:
r = requests.post(API_URL + '/predict_demo', json={'culture': 'german', 'person': 1, 'clip': 2, 'session': None}, headers={'ngrok-skip-browser-warning': '1'})
out = r.json()
print('trained on:', out['trained_on'])
print('true:', out['true_label'], '| predicted:', out['answer_3class'], '| pos vs neg:', out['answer_pos_vs_neg'])
print(out['mean_probability'])

## 6. Launch the Gradio frontend
This prints a public **gradio.live** link: open it on the projector or share it with the class.

In [ ]:
import os, importlib
os.environ['API_URL'] = LOCAL_API  # Gradio runs in the same Colab, so it calls the API locally
import app
app = importlib.reload(app)  # always use the uploaded version
app.build().launch(share=True, debug=False)